In [ ]:
codigos_df = pd.read_excel("../data/originales/2 Planilla MLE 2023.xlsx", engine='openpyxl')

# Eliminamos codigos repetidos
codigos_df = codigos_df[~codigos_df["total1"].isin([0])].copy()

# Formateamos codigos a GSSPPP
codigos_df["CODIGO_PRESTACIÓN"] = (
    codigos_df['grupo'].astype(str).str.zfill(1) +
    codigos_df['sub_grupo'].astype(str).str.zfill(2) +
    codigos_df['presta'].astype(str).str.zfill(3)
)
codigos_df = codigos_df.rename(columns={"glosa": "NOMBRE_PRESTACIÓN"})

nombre_codigos = codigos_df[["CODIGO_PRESTACIÓN", "NOMBRE_PRESTACIÓN"]]

codigos_df.to_csv("../data/Presupuestos_MLE_2023_original_limpio.csv", encoding='latin1', index=False)

In [ ]:
import dask.dataframe as dd

# # Cargar Beneficiarios con Dask
# df_beneficiarios = dd.read_csv("../data/originales/Beneficiarios Fonasa 2023.csv", encoding='latin1')

# df_beneficiarios_sin_nulos = df_beneficiarios.dropna()
# df_beneficiarios_sin_nulos = df_beneficiarios_sin_nulos.drop_duplicates()

# df_beneficiarios_sin_nulos.to_csv('../data/beneficiarios_original_limpio.csv', single_file=True)

# del df_beneficiarios_sin_nulos

# Cargar MLE con Dask
df_MLE = dd.read_csv(
    "../data/originales/export MLE 2023.csv",
    encoding='latin1',
    dtype={
        "PRESTRACIONES": "float",
        "CODIGO_PRESTACIÓN": "int",
        "MES_EMISION": "int",
        "CODIGO_BENEFICIARIO": "int",
        "MONTO_FAM": "int",
        "MONTO_COPAGO": "int"
    },
    assume_missing=True
)


# Eliminación de nulos, duplicados y renombrado de columnas
# df_MLE_sin_nulos = df_MLE.dropna()
df_MLE_limpio = df_MLE.rename(columns={'PRESTRACIONES': 'PRESTACIONES'})

df_MLE_limpio["REGION_EMISION"] = df_MLE_limpio["REGION_EMISION"].replace({"Ñuble": "De Ñuble"})

# df_MLE_limpio.to_csv('../data/MLE_2023_original_limpio.csv', single_file=True)

# df_MLE_limpio['CODIGO_PRESTACIÓN'] = df_MLE_limpio['CODIGO_PRESTACIÓN'].astype(str)

nombre_codigos = dd.read_csv("../data/Presupuestos_MLE_2023_original_limpio.csv", encoding='latin1', usecols=["CODIGO_PRESTACIÓN", "NOMBRE_PRESTACIÓN"],  dtype={"CODIGO_PRESTACIÓN": "int"},)
# nombre_codigos["CODIGO_PRESTACIÓN"] = nombre_codigos["CODIGO_PRESTACIÓN"].astype(str)

df_final = df_MLE_limpio.merge(nombre_codigos, left_on="CODIGO_PRESTACIÓN", right_on="CODIGO_PRESTACIÓN", how='left')

print(df_final.head()) 

# Guardar el DataFrame final
# df_final.to_csv('../data/MLE_2023_merged_light_limpio.csv', single_file=True)
# df_final.compute().to_csv('../data/MLE_2023_merged_limpio.csv', index=False)


   MES_EMISION  CODIGO_PRESTACIÓN             DESC_SECCION  \
0       202305             309022  Exámenes De Diagnóstico   
1       202305             302023  Exámenes De Diagnóstico   
2       202305             302023  Exámenes De Diagnóstico   
3       202305             302023  Exámenes De Diagnóstico   
4       202305             302023  Exámenes De Diagnóstico   

             DESC_ITEM  CODIGO_BENEFICIARIO TRAMO_FONASA    EDAD_TRAMO  \
0  Laboratorio Clínico             97481406            D  35 a 39 años   
1  Laboratorio Clínico             76712895            X  50 a 54 años   
2  Laboratorio Clínico             69353764            B  50 a 54 años   
3  Laboratorio Clínico             76007327            B  55 a 59 años   
4  Laboratorio Clínico             81474165            B  55 a 59 años   

     SEXO             REGION_EMISION COMUNA_EMISION  PRESTACIONES  MONTO_FAM  \
0   Mujer  Metropolitana De Santiago    La Cisterna           1.0        890   
1  Hombre             

In [2]:
print(df_final.columns) 


Index(['MES_EMISION', 'CODIGO_PRESTACIÓN', 'DESC_SECCION', 'DESC_ITEM',
       'CODIGO_BENEFICIARIO', 'TRAMO_FONASA', 'EDAD_TRAMO', 'SEXO',
       'REGION_EMISION', 'COMUNA_EMISION', 'PRESTACIONES', 'MONTO_FAM',
       'MONTO_COPAGO', 'NOMBRE_PRESTACIÓN'],
      dtype='object')


In [3]:
df_final.dtypes

MES_EMISION                    float64
CODIGO_PRESTACIÓN               object
DESC_SECCION           string[pyarrow]
DESC_ITEM              string[pyarrow]
CODIGO_BENEFICIARIO            float64
TRAMO_FONASA           string[pyarrow]
EDAD_TRAMO             string[pyarrow]
SEXO                   string[pyarrow]
REGION_EMISION         string[pyarrow]
COMUNA_EMISION         string[pyarrow]
PRESTACIONES                   float64
MONTO_FAM                      float64
MONTO_COPAGO                   float64
NOMBRE_PRESTACIÓN      string[pyarrow]
dtype: object